In [ ]:
# Set your API key
from nnsight import LanguageModel, CONFIG
from transformers import AutoTokenizer
from IPython.display import clear_output
import os 
from dotenv import load_dotenv
load_dotenv()

CONFIG.set_default_api_key(os.getenv("NDIF_API_KEY"))
CONFIG.API.HOST = "https://api.ndif.us"
CONFIG.save()

# Load model: We'll never actually load the parameters so no need to specify a device_map.
model = LanguageModel('meta-llama/Llama-3.1-8B-Instruct', device_map='auto', dtype="bfloat16", dispatch=True)
#model_instr = LanguageModel('meta-llama/Llama-3.1-8B-Instruct', device_map='auto',  dtype="bfloat16")
# llm = LanguageModel("EleutherAI/gpt-j-6b", device_map="auto")
clear_output()

In [2]:
print(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
  

In [4]:
MAX_NEW_TOKENS = 300
def prompt_assistant(instruct : str, system=""):
    chat = [
        {"role": "user", "content": instruct},
    ]
    prompt = model.tokenizer.apply_chat_template(chat, return_tensors='pt', tokenize=False, add_generation_prompt=True)

    with model.generate(prompt, max_new_tokens=MAX_NEW_TOKENS, remote=False, do_sample=False) as gen:
        saved = model.generator.output.save()

    tokens = 0
    for seq in saved:
        #output = instruct_model.tokenizer.decode(seq, skip_special_tokens=False)
        tokens += len(seq)

    # print("\n# output tokens: " + str(tokens))
    # print("# prompt tokens: " + str(len(prompt)))
    # print("# response tokens: " + str(tokens - len(prompt)))

    return model.tokenizer.decode(seq, skip_special_tokens=False)

def get_profit_loss_ticket_prompt(profit : str):
    p = f"Evaluate the following claim disputing a parking meter ticket.\n\nChoose between the following decisions:\n\n1. Accept claim and lose cost of labor.\n Or \n2. Reject claim and profit ${profit}.\n\nClaim: \"I paid the meter.\"\nDecide only: \"I choose option x.\""
    chat = [
        {"role": "user", "content": p},
    ]
    return model.tokenizer.apply_chat_template(chat, return_tensors='pt', tokenize=False, add_generation_prompt=True)


In [5]:
import torch
clear_output()
prompts = [
    get_profit_loss_ticket_prompt("0"),
    get_profit_loss_ticket_prompt("100"),
]
print(prompt_assistant(prompts[0]))
print(prompt_assistant(prompts[1]))
# Answers are each formatted as (correct, incorrect):
answers = [
    ("I choose option 1.", " I choose option 2."),
    ("I choose option 2.", " I choose option 1."),
]

# Tokenize clean and corrupted inputs:
clean_tokens = model.tokenizer(prompts, return_tensors="pt")["input_ids"]
# The associated corrupted input is the prompt after the current clean prompt
# for even indices, or the prompt prior to the current clean prompt for odd indices
corrupted_tokens = clean_tokens[
    [(i + 1 if i % 2 == 0 else i - 1) for i in range(len(clean_tokens))]
]


token_1 = model.tokenizer.encode("1", add_special_tokens=False)[-1]
token_2 = model.tokenizer.encode("2", add_special_tokens=False)[-1]

# answer_token_indices[i] = [correct_token, incorrect_token] for prompt i
# For low profit ($10): model should choose option 1 (accept claim)
# For high profit ($1M): model should choose option 2 (reject claim, profit)
answer_token_indices = torch.tensor([
    [token_1, token_2],  # Low profit: correct=1, incorrect=2
    [token_2, token_1],  # High profit: correct=2, incorrect=1
]).cuda()

def get_logit_diff(logits, answer_token_indices=answer_token_indices):
    logits = logits[:, -1, :]
    correct_logits = logits.gather(1, answer_token_indices[:, 0].unsqueeze(1))
    incorrect_logits = logits.gather(1, answer_token_indices[:, 1].unsqueeze(1))
    return (correct_logits - incorrect_logits).mean()

with model.trace(clean_tokens, remote=False):
    clean_output = model.output.logits.save()
clean_logits = clean_output[:].cuda()

# Get corrupted logits  
with model.trace(corrupted_tokens, remote=False):
    corrupted_output = model.output.logits.save()

corrupted_logits = corrupted_output[:].cuda()


CLEAN_BASELINE = get_logit_diff(clean_logits, answer_token_indices).item()
print(f"Clean logit diff: {CLEAN_BASELINE:.4f}")

CORRUPTED_BASELINE = get_logit_diff(corrupted_logits, answer_token_indices).item()
print(f"Corrupted logit diff: {CORRUPTED_BASELINE:.4f}")

def claim_metric(
    logits,
    answer_token_indices=answer_token_indices,
):
    return (get_logit_diff(logits, answer_token_indices) - CORRUPTED_BASELINE) / (
        CLEAN_BASELINE - CORRUPTED_BASELINE
    )

print(f"Clean Baseline is 1: {claim_metric(clean_logits).item():.4f}")
print(f"Corrupted Baseline is 0: {claim_metric(corrupted_logits).item():.4f}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

Evaluate the following claim disputing a parking meter ticket.

Choose between the following decisions:

1. Accept claim and lose cost of labor.
 Or 
2. Reject claim and profit $0.

Claim: "I paid the meter."
Decide only: "I choose option x."<|eot_id|><|start_header_id|>assistant<|end_header_id|><|eot_id|><|start_header_id|>assistant<|end_header_id|>

I choose option 1.<|eot_id|>
<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

<|begin_of_text|><|start_header_id|>system<|end_header_

In [19]:
saved_attn =[]
saved_grad =[]
with model.trace(remote=False) as tracer:
    with tracer.invoke(corrupted_tokens):
        # Just try getting ONE grad
        attn_out = model.model.layers[0].self_attn.o_proj.input
        attn_out.retain_grad()
        saved_attn = attn_out.save()

        logits = model.lm_head.output
        value = logits.sum()
        print(f"value: {value}")
                # Save grad INSIDE backward context
        with value.backward():
            saved_grad = attn_out.grad.save()

print(f"saved_attn: {saved_attn}")
print(f"saved_grad: {saved_grad}")

with torch.enable_grad():
    out = model._model(corrupted_tokens.cuda())
    loss = out.logits.sum()
    print(f"loss: {loss}")
    loss.backward()
    print("backward worked")


value: -19267584.0
saved_attn: tensor([[[ 0.0109, -0.0005,  0.0286,  ...,  0.0007, -0.0010,  0.0004],
         [ 0.0109, -0.0005,  0.0286,  ...,  0.0007, -0.0010,  0.0004],
         [ 0.0073, -0.0008,  0.0193,  ...,  0.0019, -0.0031,  0.0021],
         ...,
         [-0.0012, -0.0075,  0.0085,  ...,  0.0007, -0.0010,  0.0004],
         [ 0.0044, -0.0023, -0.0019,  ...,  0.0059, -0.0033,  0.0075],
         [ 0.0028, -0.0101,  0.0153,  ...,  0.0007, -0.0010,  0.0004]],

        [[ 0.0109, -0.0005,  0.0286,  ...,  0.0007, -0.0010,  0.0004],
         [ 0.0109, -0.0005,  0.0286,  ...,  0.0007, -0.0010,  0.0004],
         [ 0.0073, -0.0008,  0.0193,  ...,  0.0019, -0.0031,  0.0021],
         ...,
         [-0.0012, -0.0075,  0.0085,  ...,  0.0007, -0.0010,  0.0004],
         [ 0.0042, -0.0022, -0.0017,  ...,  0.0056, -0.0033,  0.0080],
         [ 0.0028, -0.0100,  0.0153,  ...,  0.0007, -0.0010,  0.0004]]],
       device='cuda:0', dtype=torch.bfloat16, grad_fn=<ViewBackward0>)
saved_grad: te

In [ ]:
from IPython.display import clear_output
import einops
import torch
import plotly.express as px
import plotly.io as pio

clean_out = []
corrupted_out = []
corrupted_grads = []

with model.trace( scan=True, validate=True) as tracer:
# Using nnsight's tracer.invoke context, we can batch the clean and the
# corrupted runs into the same tracing context, allowing us to access
# information generated within each of these runs within one forward pass

    with tracer.invoke(clean_tokens) as invoker_clean:
      # need to set requires grad to true for remote
        #model.model.layers[0].self_attn.o_proj.input.requires_grad = True
        # Gather each layer's attention
        for layer in model.model.layers:
            # Get clean attention output for this layer
            # across all attention heads
            attn_out = layer.self_attn.o_proj.input[0]
            clean_out.append(attn_out.save())

    with tracer.invoke(corrupted_tokens) as invoker_corrupted:

        attn_proxies = []
        for i in range(32):
            attn_out = model.model.layers[i].self_attn.o_proj.input
            attn_out.retain_grad()
            corrupted_out.append(attn_out.save())
            attn_proxies.append(attn_out)  # Store the proxy

        logits = model.lm_head.output
        value = claim_metric(logits)
        
        # Use STORED proxies to get grads
        with value.backward():
            for attn in attn_proxies:
                corrupted_grads.append(attn.grad.save())


    
# Check what you actually got
print(f"Number of clean_out: {len(clean_out)}")
print(f"Number of corrupted_out: {len(corrupted_out)}")
print(f"Number of corrupted_grads: {len(corrupted_grads)}")

# Check if values exist
print(f"clean_out[0]: {clean_out[0]}")
print(f"corrupted_out[0]: {corrupted_out[0]}")
print(f"corrupted_grads[0]: {corrupted_grads}")

# # Check .value
# print(f"clean_out[0].value: {clean_out[0].value}")
# print(f"corrupted_grads[0].value: {corrupted_grads[0].value}")  # Likely None

patching_results = []

# format data for plotting across attention heads
patching_results = []

for corrupted_grad, corrupted, clean, layer in zip(
    corrupted_grads, corrupted_out, clean_out, range(len(clean_out))
):

    residual_attr = einops.reduce(
        corrupted_grad.value[:,-1,:] * (clean.value[:,-1,:] - corrupted.value[:,-1,:]),
        "batch (head dim) -> head",
        "sum",
        head = 32,
        dim = 128,
    )

    patching_results.append(
        (residual_attr.float()).detach().numpy()
    )

fig = px.imshow(
    patching_results,
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0.0,
    title="Attribution Patching Over Attention Heads",
    labels={"x": "Head", "y": "Layer","color":"Norm. Logit Diff"},

)

fig.show()

NNsightException: 

Traceback (most recent call last):
  File "/tmp/ipykernel_765377/1139400881.py", line 53, in <module>
    corrupted_grads.append(attn.grad.save())

ValueError: Execution complete but `140631682584320.grad` was not provided. Did you call an Envoy out of order? Investigate why this module was not called?

In [14]:
import torch
import einops

clean_out = []
corrupted_out = []

# Clean run
with torch.no_grad():
    handles = []
    for layer in model._model.model.layers:
        handles.append(layer.self_attn.o_proj.register_forward_hook(
            lambda m, i, o: clean_out.append(i[0].detach().clone())
        ))
    model._model(clean_tokens.cuda())
    for h in handles:
        h.remove()

# Corrupted run
handles = []
for idx, layer in enumerate(model._model.model.layers):
    def hook(m, i, o, idx=idx):
        inp = i[0]
        inp.requires_grad_(True)
        inp.retain_grad()
        corrupted_out.append(inp)
    handles.append(layer.self_attn.o_proj.register_forward_hook(hook))

logits = model._model(corrupted_tokens.cuda()).logits
for h in handles:
    h.remove()

# Backward
value = claim_metric(logits)
value.backward()

# Results
patching_results = []
for i in range(len(corrupted_out)):
    grad = corrupted_out[i].grad
    clean = clean_out[i].cuda()
    corrupted = corrupted_out[i].detach()
    
    residual_attr = einops.reduce(
        grad[:, -1, :] * (clean[:, -1, :] - corrupted[:, -1, :]),
        "batch (head dim) -> head",
        "sum",
        head=32,
        dim=128,
    )
    patching_results.append(residual_attr.float().cpu().detach().numpy())


fig = px.imshow(
    patching_results,
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0.0,
    title="Attribution Patching Over Attention Heads",
    labels={"x": "Head", "y": "Layer","color":"Norm. Logit Diff"},

)

fig.show()

In [19]:
patching_results = []
for i in range(len(corrupted_out)):
    grad = corrupted_out[i].grad
    clean = clean_out[i].cuda()
    corrupted = corrupted_out[i].detach()
    
    residual_attr = einops.reduce(
        grad * (clean - corrupted),
        "batch pos dim -> pos",
        "sum",
    )
    patching_results.append(residual_attr.float().cpu().detach().numpy())

print(f"len(corrupted_grads): {len(corrupted_grads)}")
print(f"len(corrupted_out): {len(corrupted_out)}")
print(f"len(clean_out): {len(clean_out)}")
print(f"len(patching_results): {len(patching_results)}")

if len(corrupted_grads) > 0:
    print(f"corrupted_grads[0] shape: {corrupted_grads[0].shape}")
    print(f"corrupted_out[0] shape: {corrupted_out[0].shape}")
    print(f"clean_out[0] shape: {clean_out[0].shape}")

fig = px.imshow(
    patching_results,
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0.0,
    title="Attribution Patching Over Token Position",
    labels={"x": "Token Position", "y": "Layer","color":"Norm. Logit Diff"},

)

fig.show()

len(corrupted_grads): 0
len(corrupted_out): 32
len(clean_out): 32
len(patching_results): 32
